In [1]:
import torch
import  numpy as np
import pandas as pd
from PhiFCM import PhiFCM   

In [2]:
# Paramaters
alpha = 1.5
K = 2
m = 2.0
device = "cpu"

# Inequality index J_\alpha
def inequality_index(X, model):
    Y = model.phi(X, model.alpha)
    U = model.U
    W = model.W
    n = X.shape[0]
    q = 2.0 / model.alpha
    
    # With-cluster inequality
    dist = torch.cdist(Y, W) ** q
    within = (U**model.m * dist).sum() / n
    
    # Between-cluster inequality
    s_k = (U**model.m).sum(dim=0)
    pi_k = s_k / n
    y_bar = (pi_k[:, None] * W).sum(dim=0)
    center_dist = torch.norm(W - y_bar, dim=1) ** q
    between = (pi_k * center_dist).sum()
    total = within + between
    
    return total.item(), within.item(), between.item()

# Clustering the data
def clustering_from_FS(X, K, alpha, m):
    model = PhiFCM(K, alpha=alpha, m=m)
    model.fit(X)
    fs = model.fukuyama_sugeno(
        model.phi(X, alpha),
        model.U,
        model.W
    )
    return model, fs

# Compute inequality
def run_analysis(df, feature_cols, K, alpha, m, cluster_col=None):
    X = torch.tensor(df[feature_cols].values, dtype=torch.float32)

    # -----------------------------
    # CASE 1: cluster column given
    # -----------------------------
    if cluster_col is not None:
        labels = df[cluster_col].values
        K = len(set(labels))

        model = PhiFCM(K, alpha=alpha, m=m)

        U = torch.zeros((len(labels), K))
        for i, k in enumerate(labels):
            U[i, k] = 1.0

        model.U = U
        model.W = model.update_centroids(
            model.phi(X, alpha), U
        )

    # -----------------------------
    # CASE 2: automatic clustering
    # -----------------------------
    else:
        model, fs = clustering_from_FS(X, K, alpha, m)

    total, within, between = inequality_index(X, model)

    return {
        "Total inequality": total,
        "Within": within,
        "Between": between
    }

# Example

In [3]:
data = pd.DataFrame({
    "income": [10, 20, 8, 15],
    "cluster": [0, 0, 1, 1]
})

# Case 1: fixed clusters
res1 = run_analysis(data, ["income"], K=2, alpha=alpha, m=m, cluster_col="cluster")

# Case 2: automatic clustering
res2 = run_analysis(data, ["income"], K=2, alpha=alpha, m=m, cluster_col=None)

results_df = pd.DataFrame([
    {"Case": "Fixed clusters", **res1},
    {"Case": "Auto clustering", **res2}
])

results_df

,Case,Total inequality,Within,Between
0,Fixed clusters,88.731873,67.487823,21.244053
1,Auto clustering,80.628975,15.273417,65.355560


# German credit example on fixed clusters: foreign worker 0/1

In [4]:
from ucimlrepo import fetch_ucirepo
from sklearn.preprocessing import StandardScaler

# Load dataset
data = fetch_ucirepo(id=144)
X_full = data.data.features
y = data.data.targets

# Cluster variables
cols_idx = [1, 4, 7, 10, 12, 15, 17]   # numerical vars
cluster_idx = 19                       # foreign worker
df = X_full.iloc[:, cols_idx + [cluster_idx]].copy()
df["Attribute20"] = df["Attribute20"].map({
    "A201": 1,
    "A202": 0
})

# Numerical variables
cols_num = df.columns[:-1].tolist()

# pre-processing (scaler already incude in the .py)

# First application: CLuster is given by "Foreign worker" 0/1
alphas_1 = [1.0, 1.5, 2.0]
results_sim1 = []
for alpha in alphas_1:
    res = run_analysis(
        df,
        cols_num,
        K=2,
        alpha=alpha,
        m=2.0,
        cluster_col="Attribute20"
    )
    results_sim1.append({
        "alpha": alpha,
        **res
    })

df_sim1 = pd.DataFrame(results_sim1)
df_sim1.round(4)

,alpha,Total inequality,Within,Between
0,1.0,7960152.0,7940209.5,19942.3203
1,1.5,13856159.0,13729524.0,126634.9141
2,2.0,17015030.0,16649753.0,365276.5625


# German credit example with clustering the dataset with FS

In [5]:
def optimize_m(X, K, alpha, m_grid):
    best_m = None
    best_fs = np.inf
    best_model = None

    for m in m_grid:
        model = PhiFCM(K, alpha=alpha, m=m)
        model.fit(X)

        fs = model.fukuyama_sugeno(
            model.phi(X, alpha),
            model.U,
            model.W
        )

        if fs < best_fs:
            best_fs = fs
            best_m = m
            best_model = model

    return best_m, best_model

alphas_2 = [1.0, 1.5, 2.0]
m_grid = [1.5, 2.0, 2.5, 3.0]

results_sim2 = []
cluster_stats = {}

X_tensor = torch.tensor(df.values, dtype=torch.float32)

for alpha in alphas_2:
    best_m, model = optimize_m(X_tensor, K=2, alpha=alpha, m_grid=m_grid)
    total, within, between = inequality_index(X_tensor, model)
    results_sim2.append({
        "alpha": alpha,
        "optimal_m": best_m,
        "Total": total,
        "Within": within,
        "Between": between
    })

    # cluster labels
    labels = model.labels_.numpy()
    df_tmp = df.copy()
    df_tmp["cluster"] = labels

    # descriptive stats
    cluster_stats[alpha] = df_tmp.groupby("cluster").mean()

df_sim2 = pd.DataFrame(results_sim2)
print(df_sim2.round(4))
cluster_stats


   alpha  optimal_m       Total      Within    Between
0    1.0        1.5   7681223.0  2293266.75  5387956.5
1    1.5        1.5  15333948.0  6052066.00  9281882.0
2    2.0        1.5  18683496.0  9164711.00  9518784.0


{1.0:          Attribute2   Attribute5  Attribute8  Attribute11  Attribute13  \
 cluster                                                                  
 0         17.960000  2185.780606    3.077576     2.832727    35.273939   
 1         34.777143  8388.508571    2.480000     2.902857    36.828571   
 
          Attribute16  Attribute18  Attribute20  
 cluster                                         
 0           1.392727     1.152727     0.958788  
 1           1.474286     1.165714     0.982857  ,
 1.5:          Attribute2   Attribute5  Attribute8  Attribute11  Attribute13  \
 cluster                                                                  
 0         17.960000  2185.780606    3.077576     2.832727    35.273939   
 1         34.777143  8388.508571    2.480000     2.902857    36.828571   
 
          Attribute16  Attribute18  Attribute20  
 cluster                                         
 0           1.392727     1.152727     0.958788  
 1           1.474286     1.165714 

In [6]:
pd.crosstab(df_tmp["Attribute20"],df_tmp["cluster"])

cluster,0,1
Attribute20,,
0,3,34
1,186,777


# Example of transfers with the .py program

In [9]:
X_before = torch.tensor([
    [10.0, 10.0],
    [20.0, 10.0],
    [8.0,  10.0],
    [15.0, 10.0]
])
X_after = torch.tensor([
    [11.0, 10.0],
    [19.0, 10.0],
    [8.0,  10.0],
    [15.0, 10.0]
])
clusters = torch.tensor([0, 0, 1, 1])


# Build fixed clusters
def build_U(clusters, K):
    n = len(clusters)
    U = torch.zeros(n, K)
    for i in range(n):
        U[i, clusters[i]] = 1.0
    return U


# Inequality index
def inequality_index(X, model):
    Y = model.phi(X, model.alpha)
    U = model.U
    W = model.W
    n = X.shape[0]
    q = 2.0 / model.alpha
    dist = torch.cdist(Y, W) ** q
    within = (U**model.m * dist).sum() / n
    
    s_k = (U**model.m).sum(dim=0)
    pi_k = s_k / n
    y_bar = (pi_k[:, None] * W).sum(dim=0)
    center_dist = torch.norm(W - y_bar, dim=1) ** q
    
    between = (pi_k * center_dist).sum()
    total = within + between

    return total.item(), within.item(), between.item()


# Compute index with FIXED U and FIXED V
def compute_index_fixed_V(X, U, W, alpha, m):
    model = PhiFCM(k=U.shape[1], alpha=alpha, m=m)
    model.U = U
    model.W = W
    total, within, between = inequality_index(X, model)
    return total, within, between


# Transfer example
alphas = [1, 2, 3, 4]
m = 2.0
K = 2
U = build_U(clusters, K)

results = []
for alpha in alphas:
    model_tmp = PhiFCM(k=K, alpha=alpha, m=m)
    Y_before = model_tmp.phi(X_before, alpha)
    W_fixed = model_tmp.update_centroids(Y_before, U)

    # evaluate BEFORE and AFTER with SAME W
    total_b, _, _ = compute_index_fixed_V(X_before, U, W_fixed, alpha, m)
    total_a, _, _ = compute_index_fixed_V(X_after,  U, W_fixed, alpha, m)

    results.append({
        "alpha": alpha,
        "Before": total_b,
        "After": total_a,
        "Variation": total_a - total_b
    })

df_fixed = pd.DataFrame(results)

print("Fixed U and V (true PD test):")
print(df_fixed.round(4))

Fixed U and V (true PD test):
   alpha    Before     After  Variation
0      1   21.6875   17.1875    -4.5000
1      2  115.2500  100.2500   -15.0000
2      3  254.0392  255.7582     1.7190
3      4  293.2729  300.1979     6.9249
